# Financial Risk Model Tutorial

This notebook demonstrates how to use the Financial Risk Modeling system in JupyterLab.

## Overview

The system combines:
- **Transformer**: For temporal pattern recognition in financial time series
- **GNN (Graph Neural Network)**: For modeling entity relationships
- **SHAP**: For interpretable predictions

## Table of Contents
1. [Setup](#setup)
2. [Generate Sample Data](#generate-sample-data)
3. [Data Exploration](#data-exploration)
4. [Load and Preprocess Data](#load-and-preprocess)
5. [Create Model](#create-model)
6. [Train Model](#train-model)
7. [Evaluate Model](#evaluate-model)
8. [Make Predictions](#make-predictions)
9. [SHAP Explanations](#shap-explanations)
10. [Custom Data](#custom-data)

## 1. Setup <a name="setup"></a>

First, let's import the necessary libraries and check if GPU is available.

In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

# Import project modules
from config import Config, set_seed
from data_loader import load_and_preprocess_data, create_data_loaders
from models.hybrid import HybridRiskModel
from train import Trainer
from utils import calculate_metrics, SHAPExplainer

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Check device
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {Config.DEVICE}")

## 2. Generate Sample Data <a name="generate-sample-data"></a>

Let's generate synthetic financial data for testing.

In [ ]:
# Generate sample data
from data.generate_sample_data import generate_sample_data

# Generate data with 50 entities, 60 time steps, 10 features
df = generate_sample_data(
    num_entities=50,
    sequence_length=60,
    num_features=10,
    output_path='./data/sample_data.csv'
)

print("\nFirst few rows:")
print(df.head(10))

## 3. Data Exploration <a name="data-exploration"></a>

Let's explore the generated data.

In [ ]:
# Load data
df = pd.read_csv('./data/sample_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nUnique entities: {df['entity_id'].nunique()}")
print(f"\nLabel distribution:")
print(df.groupby('entity_id')['label'].first().value_counts())

# Visualize price trajectories
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot a few entities from each class
low_risk_entities = df[df['label'] == 0]['entity_id'].unique()[:3]
high_risk_entities = df[df['label'] == 1]['entity_id'].unique()[:3]

for entity_id in low_risk_entities:
    entity_data = df[df['entity_id'] == entity_id]
    axes[0].plot(entity_data['time_step'], entity_data['price'], alpha=0.7, label=f'Entity {entity_id}')
axes[0].set_title('Low Risk Entities - Price Trajectories')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Price')
axes[0].legend()
axes[0].grid(True)

for entity_id in high_risk_entities:
    entity_data = df[df['entity_id'] == entity_id]
    axes[1].plot(entity_data['time_step'], entity_data['price'], alpha=0.7, label=f'Entity {entity_id}')
axes[1].set_title('High Risk Entities - Price Trajectories')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Price')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 4. Load and Preprocess Data <a name="load-and-preprocess"></a>

Load the data and create graph structure.

In [ ]:
# Set random seed for reproducibility
set_seed(42)

# Load and preprocess data
temporal_data, graph_data, labels = load_and_preprocess_data(
    data_path='./data/sample_data.csv',
    sequence_length=Config.SEQUENCE_LENGTH,
    num_features=Config.NUM_FEATURES,
    graph_method='correlation',
    threshold=0.5  # Lower threshold to create more edges
)

print(f"Temporal data shape: {temporal_data.shape}")
print(f"  - Samples: {temporal_data.shape[0]}")
print(f"  - Sequence length: {temporal_data.shape[1]}")
print(f"  - Features: {temporal_data.shape[2]}")
print(f"\nGraph data:")
print(f"  - Nodes: {graph_data.x.shape[0]}")
print(f"  - Node features: {graph_data.x.shape[1]}")
print(f"  - Edges: {graph_data.edge_index.shape[1]}")
print(f"\nLabels shape: {labels.shape}")
print(f"Label distribution: {np.bincount(labels)}")

## 5. Create Model <a name="create-model"></a>

Initialize the Hybrid Risk Model.

In [ ]:
# Update config for faster training (optional)
Config.NUM_EPOCHS = 20
Config.BATCH_SIZE = 8
Config.TRANSFORMER_LAYERS = 2
Config.GNN_LAYERS = 2

# Create model
model = HybridRiskModel(
    num_temporal_features=Config.NUM_FEATURES,
    sequence_length=Config.SEQUENCE_LENGTH,
    transformer_dim=Config.TRANSFORMER_DIM,
    transformer_heads=Config.TRANSFORMER_HEADS,
    transformer_layers=Config.TRANSFORMER_LAYERS,
    transformer_ff_dim=Config.TRANSFORMER_FF_DIM,
    transformer_dropout=Config.TRANSFORMER_DROPOUT,
    num_node_features=graph_data.x.shape[1],
    gnn_hidden_dim=Config.GNN_HIDDEN_DIM,
    gnn_layers=Config.GNN_LAYERS,
    gnn_dropout=Config.GNN_DROPOUT,
    gnn_type=Config.GNN_TYPE,
    gnn_heads=Config.GNN_HEADS,
    fusion_method=Config.FUSION_METHOD,
    hybrid_hidden_dim=Config.HYBRID_HIDDEN_DIM,
    hybrid_dropout=Config.HYBRID_DROPOUT,
    num_classes=Config.NUM_CLASSES
)

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model created successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:")
print(model)

## 6. Train Model <a name="train-model"></a>

Train the model on the data.

In [ ]:
# Create data loaders
train_loader, val_loader, test_loader = create_data_loaders(
    temporal_data,
    graph_data,
    labels,
    batch_size=Config.BATCH_SIZE,
    train_split=0.7,
    val_split=0.15,
    test_split=0.15,
    random_seed=42
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Create trainer
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    config=Config
)

# Train the model
print("Starting training...\n")
history = trainer.train()

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBest Validation Accuracy: {max(history['val_acc']):.2f}%")
print(f"Final Validation Loss: {history['val_loss'][-1]:.4f}")

## 7. Evaluate Model <a name="evaluate-model"></a>

Evaluate the trained model on the test set.

In [ ]:
# Evaluate on test set
test_metrics = trainer.evaluate(save_dir='./results')

print("\nTest Set Results:")
print("=" * 50)
for metric, value in test_metrics.items():
    if metric == 'accuracy':
        print(f"{metric.capitalize()}: {value*100:.2f}%")
    else:
        print(f"{metric.upper()}: {value:.4f}")

In [ ]:
# Display confusion matrix
if os.path.exists('./results/confusion_matrix.png'):
    print("\nConfusion Matrix:")
    display(Image(filename='./results/confusion_matrix.png'))

## 8. Make Predictions <a name="make-predictions"></a>

Use the trained model to make predictions.

In [ ]:
# Make predictions on test set
model.eval()
all_predictions = []
all_probabilities = []
all_labels = []

with torch.no_grad():
    for temporal_data, graph_data, labels in test_loader:
        temporal_data = temporal_data.to(Config.DEVICE)
        graph_data = graph_data.to(Config.DEVICE)
        
        outputs = model(temporal_data, graph_data)
        probabilities = torch.softmax(outputs, dim=1)
        predictions = outputs.argmax(dim=1)
        
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())
        all_labels.extend(labels.numpy())

all_predictions = np.array(all_predictions)
all_probabilities = np.array(all_probabilities)
all_labels = np.array(all_labels)

# Create results DataFrame
results_df = pd.DataFrame({
    'True Label': all_labels,
    'Predicted Label': all_predictions,
    'Probability Low Risk': all_probabilities[:, 0],
    'Probability High Risk': all_probabilities[:, 1],
    'Correct': all_labels == all_predictions
})

print("\nSample Predictions:")
print(results_df.head(10))

print(f"\nOverall Accuracy: {(all_labels == all_predictions).mean() * 100:.2f}%")

In [ ]:
# Visualize prediction confidence
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Correct predictions
correct_probs = results_df[results_df['Correct']]['Probability High Risk']
axes[0].hist(correct_probs, bins=20, alpha=0.7, color='green', edgecolor='black')
axes[0].set_xlabel('Predicted Probability (High Risk)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Prediction Confidence - Correct Predictions', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Incorrect predictions
incorrect_probs = results_df[~results_df['Correct']]['Probability High Risk']
axes[1].hist(incorrect_probs, bins=20, alpha=0.7, color='red', edgecolor='black')
axes[1].set_xlabel('Predicted Probability (High Risk)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Prediction Confidence - Incorrect Predictions', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. SHAP Explanations <a name="shap-explanations"></a>

Generate SHAP explanations to understand model predictions.

In [ ]:
# Get sample data for SHAP
sample_temporal = []
sample_graph = None

for temporal_data, graph_data, _ in test_loader:
    sample_temporal.append(temporal_data)
    if sample_graph is None:
        sample_graph = graph_data
    if len(sample_temporal) * temporal_data.size(0) >= 20:
        break

sample_temporal = torch.cat(sample_temporal, dim=0)[:20]

print(f"Sample data for SHAP: {sample_temporal.shape}")

In [ ]:
# Create SHAP explainer
print("Creating SHAP explainer...")
explainer = SHAPExplainer(
    model,
    (sample_temporal[:10], sample_graph),
    Config.DEVICE
)

# Generate SHAP values
print("Generating SHAP values...")
test_temporal = sample_temporal[10:15]
shap_values, base_values = explainer.explain_instance(test_temporal, sample_graph)

print("SHAP values generated!")

In [ ]:
# Plot SHAP summary
explainer.plot_summary(
    shap_values,
    test_temporal.cpu().numpy(),
    save_path='./results/shap_summary.png'
)

if os.path.exists('./results/shap_summary.png'):
    print("\nSHAP Summary Plot:")
    display(Image(filename='./results/shap_summary.png'))

In [ ]:
# Plot SHAP waterfall for a single prediction
explainer.plot_waterfall(
    shap_values,
    test_temporal.cpu().numpy(),
    instance_idx=0,
    save_path='./results/shap_waterfall.png'
)

if os.path.exists('./results/shap_waterfall.png'):
    print("\nSHAP Waterfall Plot (First Instance):")
    display(Image(filename='./results/shap_waterfall.png'))

## 10. Using Custom Data <a name="custom-data"></a>

Here's how to use your own financial data.

In [ ]:
# Example: Load your custom data
# Make sure your data follows this format:

custom_data_example = pd.DataFrame({
    'entity_id': [0, 0, 0, 1, 1, 1],
    'time_step': [0, 1, 2, 0, 1, 2],
    'feature_1': [0.5, 0.6, 0.4, -0.2, -0.3, -0.1],
    'feature_2': [1.2, 1.1, 1.3, 0.8, 0.9, 0.7],
    'price': [100.0, 101.5, 99.8, 50.0, 49.5, 51.2],
    'label': [0, 0, 0, 1, 1, 1]
})

print("Custom Data Format Example:")
print(custom_data_example)

print("\nTo use your own data:")
print("1. Prepare your data in the format above")
print("2. Save it as a CSV file")
print("3. Update the data_path in the load_and_preprocess_data function")
print("4. Adjust Config.NUM_FEATURES and Config.SEQUENCE_LENGTH as needed")
print("5. Run the training pipeline")

## Summary

You've successfully:
1. ✅ Generated synthetic financial data
2. ✅ Loaded and preprocessed the data
3. ✅ Created a Hybrid Risk Model (Transformer + GNN)
4. ✅ Trained the model
5. ✅ Evaluated performance
6. ✅ Made predictions
7. ✅ Generated SHAP explanations

### Next Steps

- Fine-tune hyperparameters in `config.py`
- Try different graph construction methods
- Experiment with different model architectures
- Use your own financial data
- Save and load models for production use

### Saving and Loading Models

```python
# Save model
torch.save(model.state_dict(), './checkpoints/my_model.pth')

# Load model
model = HybridRiskModel(...)
model.load_state_dict(torch.load('./checkpoints/my_model.pth'))
model.eval()
```

### Additional Resources

- See `README.md` for detailed documentation
- Check `data/README.md` for data format specifications
- See `results/README.md` for output explanations